# CatBoost + Optuna

In [1]:
import warnings
import pandas as pd
import optuna

from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import f1_score, roc_auc_score

from catboost import CatBoostClassifier


# 경고 숨김부
warnings.filterwarnings("ignore")

# 결과 출력 형식 설정부
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

# 파일 경로 설정부
train_valid_file_path = r"C:\Users\Dell3571\Desktop\vscode\study\SKAX\ott-churn-prediction\ott-churn-prediction\kim.kwangil\data\03_1차 파생\260602_최종파생변수(80개).csv"
test_file_path = r"C:\Users\Dell3571\Desktop\vscode\study\SKAX\ott-churn-prediction\ott-churn-prediction\kim.kwangil\data\03_1차 파생\260602_테스트용(80개).csv"

# 모델 입력 제외 컬럼 설정부
exclude_cols = [
    "USER_KEY",
    "product_code",
    "price",
    "billing_method",
    "payment_device",
    "gender",
    "age",
    "reg_date",
    "reg_hour",
    "end_date",
    "is_repurchase",
]


# 이진 변수 변환 함수부
def to_binary(series):
    lowered = series.astype(str).str.strip().str.lower()

    mapped = lowered.map(
        {
            "1": 1,
            "0": 0,
            "y": 1,
            "n": 0,
            "yes": 1,
            "no": 0,
            "true": 1,
            "false": 0,
        }
    )

    numeric = pd.to_numeric(series, errors="coerce")

    return mapped.where(mapped.notna(), numeric)


# OneHotEncoder 버전 호환 함수부
def make_onehot_encoder():
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=False,
        )


# 타깃 분포 출력 함수부
def print_target_distribution(y):
    target_dist = (
        pd.DataFrame({"count": y.value_counts().sort_index()})
        .assign(percent=lambda x: (x["count"] / x["count"].sum() * 100).round(2))
    )

    print(target_dist)


# 전처리 파이프라인 생성 함수부
def make_preprocessor(numeric_features, categorical_features):
    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", make_onehot_encoder()),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features),
        ]
    )

    return preprocessor


# 데이터 전처리 함수부
def prepare_data(df, selected_features, numeric_features):
    df = df.copy()

    for col in numeric_features:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df["is_repurchase_num"] = to_binary(df["is_repurchase"])
    df = df[df["is_repurchase_num"].isin([0, 1])].copy()

    X = df[selected_features].copy()
    y = (df["is_repurchase_num"] == 0).astype(int)

    return X, y


# 데이터 로드부
train_valid_df = pd.read_csv(train_valid_file_path).copy()
test_df = pd.read_csv(test_file_path).copy()

# 전체 컬럼 기준 입력 변수 생성부
selected_features = [
    col for col in train_valid_df.columns if col not in exclude_cols
]

# 범주형 변수 설정부
categorical_features = [
    "age_group",
]

# 숫자형 변수 설정부
numeric_features = [
    col for col in selected_features if col not in categorical_features
]

# 입력 변수, 타깃 변수 생성부
X_train_valid, y_train_valid = prepare_data(
    train_valid_df,
    selected_features,
    numeric_features,
)

X_test, y_test = prepare_data(
    test_df,
    selected_features,
    numeric_features,
)

print("분석 기준: 최종 파생 변수 + CatBoost Optuna 5-Fold 튜닝 (과적합 페널티 적용)")
print("양성 클래스 기준: is_repurchase == 0")
print(f"train/valid 데이터 수: {len(X_train_valid)}")
print(f"test 데이터 수: {len(X_test)}")
print(f"전체 데이터 수: {len(X_train_valid) + len(X_test)}")
print(f"train/valid 비율: {len(X_train_valid) / (len(X_train_valid) + len(X_test)) * 100:.2f}%")
print(f"test 비율: {len(X_test) / (len(X_train_valid) + len(X_test)) * 100:.2f}%")
print(f"train/valid 파일 전체 컬럼 수: {len(train_valid_df.columns)}")
print(f"test 파일 전체 컬럼 수: {len(test_df.columns)}")
print(f"모델 입력 제외 컬럼 수: {len(exclude_cols)}")
print(f"사용 변수 수: {len(selected_features)}")
print("사용 변수")
print(selected_features)
print("train/valid 타깃 분포")
print_target_distribution(y_train_valid)
print("test 타깃 분포")
print_target_distribution(y_test)

if y_train_valid.nunique() < 2 or y_train_valid.value_counts().min() < 5:
    print("학습 불가: train/valid 타깃 클래스가 부족합니다.")
elif y_test.nunique() < 2:
    print("평가 불가: test 타깃 클래스가 부족합니다.")
else:
    # Optuna 목적 함수부
    # 과적합 방지 전략: 각 fold에서 train/valid 점수를 모두 계산하고
    # train-valid gap이 3%를 초과하면 초과분만큼 목적함수 점수를 차감한다.
    def objective(trial):
        params = {
            "iterations": trial.suggest_int("iterations", 200, 1000),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
            "depth": trial.suggest_int("depth", 3, 8),
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 30.0, log=True),
            "random_strength": trial.suggest_float("random_strength", 0.1, 20.0, log=True),
            "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 10.0),
            "border_count": trial.suggest_int("border_count", 32, 255),
            "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 1, 50),
            "auto_class_weights": "Balanced",
            "loss_function": "Logloss",
            "eval_metric": "AUC",
            "random_state": 42,
            "verbose": 0,
            "allow_writing_files": False,
        }

        skf = StratifiedKFold(
            n_splits=5,
            shuffle=True,
            random_state=42,
        )

        fold_scores = []

        for train_idx, valid_idx in skf.split(X_train_valid, y_train_valid):
            X_fold_train = X_train_valid.iloc[train_idx]
            X_fold_valid = X_train_valid.iloc[valid_idx]
            y_fold_train = y_train_valid.iloc[train_idx]
            y_fold_valid = y_train_valid.iloc[valid_idx]

            clf = Pipeline(
                steps=[
                    ("preprocessor", make_preprocessor(numeric_features, categorical_features)),
                    ("model", CatBoostClassifier(**params)),
                ]
            )

            clf.fit(X_fold_train, y_fold_train)

            train_proba = clf.predict_proba(X_fold_train)[:, 1]
            valid_proba = clf.predict_proba(X_fold_valid)[:, 1]

            train_roc_auc = roc_auc_score(y_fold_train, train_proba)
            valid_roc_auc = roc_auc_score(y_fold_valid, valid_proba)

            # train-valid gap이 3%를 초과하면 과적합으로 판단하고 페널티 부여
            overfit_gap = max(0.0, train_roc_auc - valid_roc_auc - 0.03)
            fold_score = valid_roc_auc - overfit_gap

            fold_scores.append(fold_score)

        return sum(fold_scores) / len(fold_scores)

    # Optuna 튜닝 수행부
    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=42),
    )

    study.optimize(
        objective,
        n_trials=200,
        show_progress_bar=True,
    )

    print("Optuna best 5-fold 목적함수 점수 (valid roc_auc - 과적합 페널티)")
    print(round(study.best_value, 4))
    print("Optuna best params")
    print(study.best_params)

    # 최적 파라미터 기반 최종 모델 학습부
    best_params = {
        **study.best_params,
        "auto_class_weights": "Balanced",
        "loss_function": "Logloss",
        "eval_metric": "AUC",
        "random_state": 42,
        "verbose": 0,
        "allow_writing_files": False,
    }

    print("최종 학습에 사용한 CatBoost 파라미터")
    print(best_params)

    final_clf = Pipeline(
        steps=[
            ("preprocessor", make_preprocessor(numeric_features, categorical_features)),
            ("model", CatBoostClassifier(**best_params)),
        ]
    )

    final_clf.fit(X_train_valid, y_train_valid)

    y_train_valid_pred = final_clf.predict(X_train_valid)
    y_test_pred = final_clf.predict(X_test)

    y_train_valid_proba = final_clf.predict_proba(X_train_valid)[:, 1]
    y_test_proba = final_clf.predict_proba(X_test)[:, 1]

    train_valid_roc_auc = roc_auc_score(y_train_valid, y_train_valid_proba)
    test_roc_auc = roc_auc_score(y_test, y_test_proba)
    auc_gap = train_valid_roc_auc - test_roc_auc

    result = {
        "model": "CatBoost_Optuna_5Fold",
        "f1_score": f1_score(y_test, y_test_pred, zero_division=0),
        "train_valid_roc_auc": train_valid_roc_auc,
        "test_roc_auc": test_roc_auc,
        "auc_gap": auc_gap,
        "is_overfit": auc_gap >= 0.05,
    }

    results_df = (
        pd.DataFrame([result])
        .set_index("model")
        [
            [
                "f1_score",
                "train_valid_roc_auc",
                "test_roc_auc",
                "auc_gap",
                "is_overfit",
            ]
        ]
        .round(4)
    )

    print("최종 테스트 성능")
    print(results_df.to_string())


[I 2026-06-04 21:50:05,882] A new study created in memory with name: no-name-789df597-de80-4cfc-9614-25fb6481868d


분석 기준: 최종 파생 변수 + CatBoost Optuna 5-Fold 튜닝 (과적합 페널티 적용)
양성 클래스 기준: is_repurchase == 0
train/valid 데이터 수: 23081
test 데이터 수: 2545
전체 데이터 수: 25626
train/valid 비율: 90.07%
test 비율: 9.93%
train/valid 파일 전체 컬럼 수: 91
test 파일 전체 컬럼 수: 91
모델 입력 제외 컬럼 수: 11
사용 변수 수: 80
사용 변수
['is_promotion', 'is_churn_prevented', 'is_user_verified', 'is_basic', 'is_standard', 'is_premium', 'age_group', 'is_female', 'is_male', 'payment_is_mobile', 'payment_is_pc', 'payment_is_android', 'payment_is_ios', 'reg_is_weekend', 'reg_hour_morning', 'reg_hour_afternoon', 'reg_hour_evening', 'reg_hour_night', 'total_watch_count', 'unique_movie', 'watch_days', 'total_watch_time(min)', 'active_ratio', 'watch_per_day', 'avg_watch_time(min)', 'median_watch_time(min)', 'std_watch_time(min)', 'max_watch_time(min)', 'avg_daily_watch_time(min)', 'max_daily_watch_time(min)', 'max_daily_sessions', 'recency', 'avg_gap_between_watch_days', 'max_inactive_gap_days', 'avg_gap_w1_watch_days', 'avg_gap_w2_watch_days', 'avg_gap_w3_watch_day

  0%|          | 0/200 [00:00<?, ?it/s]

[I 2026-06-04 21:50:47,855] Trial 0 finished with value: 0.784298873229785 and parameters: {'iterations': 500, 'learning_rate': 0.13125830316209655, 'depth': 7, 'l2_leaf_reg': 7.6611007077713635, 'random_strength': 0.22856175997064762, 'bagging_temperature': 1.5599452033620265, 'border_count': 45, 'min_data_in_leaf': 44}. Best is trial 0 with value: 0.784298873229785.
[I 2026-06-04 21:51:24,114] Trial 1 finished with value: 0.8803891928705718 and parameters: {'iterations': 681, 'learning_rate': 0.06803900745073706, 'depth': 3, 'l2_leaf_reg': 27.08160864249967, 'random_strength': 8.23143373099555, 'bagging_temperature': 2.1233911067827616, 'border_count': 72, 'min_data_in_leaf': 10}. Best is trial 1 with value: 0.8803891928705718.
[I 2026-06-04 21:51:49,226] Trial 2 finished with value: 0.879741721999098 and parameters: {'iterations': 443, 'learning_rate': 0.04141536110538596, 'depth': 5, 'l2_leaf_reg': 2.6926552514864723, 'random_strength': 2.5579488960947363, 'bagging_temperature': 1.